<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/wnut17-robust-ner/notebooks/wnut17_multi_model_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -q seqeval

In [ ]:
import sys
import numpy as np
import torch
from torch import nn
import transformers
from transformers import (AutoTokenizer,
                          AutoModelForTokenClassification,
                          DataCollatorForTokenClassification)

import datasets
from seqeval.metrics import classification_report, accuracy_score
from seqeval.scheme import IOB2

In [ ]:
wnut = datasets.load_dataset("flaitenberger/wnut_17")

In [ ]:
label_list = [label for label in wnut["train"].features["ner_tags"].feature.names]
id2label =   {i:label for i, label in enumerate(label_list)}
label2id =   {label:i for i, label in id2label.items()}


In [ ]:
shown = 0
for ex in wnut["train"]:

  if any(t != 0 for t in ex["ner_tags"]):
    print(" ".join(ex["tokens"]))
    print([(tok, id2label[t]) for tok, t in zip(ex["tokens"], ex["ner_tags"])])
    print()
  shown += 1

  if shown == 5:
    break

In [ ]:
for split in wnut:
  lens = [len(ex["tokens"]) for ex in wnut[split]]
  print(split, "mean:",round(np.mean(lens),2), "max: ", np.max(lens), "median: ", np.median(lens), "95th pct: ", np.quantile(lens, 0.95))

In [ ]:
from collections import Counter

for split in wnut:
  c = Counter(id2label[t] for ex in wnut[split] for t in ex["ner_tags"])
  ents = {l[2:]: n for l , n in c.items() if l.startswith("B-")}
  total = sum(c.values())
  print(split, "tokens:", total, "O share:", round(c["O"] / total, 3))
  print(ents, "total entities:", sum(ents.values()))

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-cased")

words = ["Pxleyes", "Top", "50", "Photography", "Contest", "#photography"]

enc = tok(words, is_split_into_words= True)

print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

In [ ]:
def tokenize_and_align(batch):

  enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):
    prev = None,
    labs = []
    for w in enc.word_ids(batch_index=i):
      if w is None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)
  enc["labels"] = all_labels

  return enc

In [ ]:
tokenized = wnut.map(tokenize_and_align, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized[split]["labels"]))
  print(split, "max_tokens:", max(len(x) for x in tokenized[split]["input_ids"]), "words lost:", lost)

In [ ]:
def evaluate_ner(gold, pred):

    rep = classification_report(
        gold,
        pred,
        mode="strict",
        scheme=IOB2,
        output_dict=True,
        zero_division=0)

    out = {"precision": rep["micro avg"]["precision"],
           "recall": rep["micro avg"]["recall"],
           "f1": rep["micro avg"]["f1-score"],
           "token_acc": accuracy_score(gold, pred)}

    out["per_type_f1"] = {k: v["f1-score"] for k, v in rep.items() if "avg" not in k}

    return out

In [ ]:
val_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]
test_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["test"]]

In [ ]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(label_list), id2label=id2label, label2id=label2id)
print(model.classifier)
print(sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))

collator = DataCollatorForTokenClassification(tok)
batch = collator([tokenized["train"][i] for i in range(3)])
print({k: v.shape for k, v in batch.items()})
print(batch["labels"][0])

In [ ]:
def compute_metrics(p):
  logits, labels = p
  preds = np.argmax(logits, axis = -1)
  gold, pred = [], []

  for pr, lb in zip(preds , labels):
    gold.append(id2label[l] for l in lb if l != -100)
    pred.append(id2label[q] for (q, l) in zip(pr, lb) if lb != -100)
  r = evaluate_ner()
  return {"precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "accuracy": r["token_acc"]}